# **Il modulo `socketserver`**

Nel notebook *programmazione_socket.ipynb* sono stati forniti esempi di implementazione di server TCP e UDP usando direttamente il modulo `socket`.

Python fornisce anche moduli e classi per scrivere server a un livello **più alto** rispetto all'uso diretto dei socket "grezzi". Il modulo principale è **`socketserver`**, che permette di implementare server **TCP e UDP sincroni**.
Un server realizzato con `socketserver`, tuttavia, è in grado di accettare più connessioni contemporaneamente se, alle classi base messe a disposizione dal modulo, si aggiungono anche funzionalità di **threading o forking** (tramite opportune classi *mix-in*).

Anche in questo notebook i server verranno eseguiti come thread *daemon*, con lo stesso approccio già usato in precedenza (porte 50000+, `allow_reuse_address`).

## 1. **Perché un livello più alto**

Il modulo **`socketserver`** semplifica il compito di scrivere server di rete definendo classi "pronte all'uso" che gestiscono le richieste su TCP, UDP (e, più raramente, gli Unix stream e datagram socket).

⇒ queste classi processano le richieste **in modo sincrono**: ogni richiesta deve essere completata prima che possa iniziare la successiva.

Per permettere al server di processare più richieste contemporaneamente, la soluzione è creare un **processo o un thread separato** per gestire ciascuna richiesta: le classi mix-in **`ForkingMixIn`** e **`ThreadingMixIn`** servono proprio a questo.

> In un linguaggio di programmazione orientato agli oggetti, una **mixin** è una classe che fornisce dei metodi ad altre classi senza doverne essere la superclasse "concettuale": è un meccanismo per iniettare del codice in una classe senza che essa erediti da quella mixin nel senso tradizionale.

## 2. **Due responsabilità distinte**

Nel modulo `socketserver`, la responsabilità di gestire una richiesta è divisa tra:

- una **classe server**, che si occupa degli aspetti di **comunicazione**: mettersi in ascolto su un socket, accettare connessioni, ecc.;
- una **classe request handler**, che si occupa degli aspetti di **protocollo**: interpretare le richieste in arrivo, elaborarle e inviare i dati di risposta al client.

⇒ questa separazione fa sì che, nella maggior parte dei casi, si possa semplicemente **usare una delle classi server già pronte senza modificarla**, fornendole una classe request handler che implementa il proprio protocollo applicativo.

## 3. **Le classi server disponibili**

Il modulo `socketserver` definisce cinque classi server diverse:

```
                +------------+
                | BaseServer |
                +------------+
                      |
                      v
   +-----------+          +------------------+
   | TCPServer |--------->| UnixStreamServer |
   +-----------+          +------------------+
         |
         v
   +-----------+          +--------------------+
   | UDPServer |--------->| UnixDatagramServer |
   +-----------+          +--------------------+
```

**`BaseServer`**: definisce l'API comune e **non** va istanziata e usata direttamente. Da essa derivano:
    - **`TCPServer`**: usa socket TCP, che forniscono flussi continui di dati tra client e server;
    - **`UDPServer`**: usa datagrammi, cioè pacchetti discreti di informazione che possono arrivare fuori ordine o andare persi durante il transito;
    - **`UnixStreamServer`** e **`UnixDatagramServer`**: le classi equivalenti che usano socket di dominio Unix, disponibili solo su piattaforme Unix (--> qui non verranno trattate).

→ tutte queste classi server hanno **gli stessi metodi e attributi esterni**, indipendentemente dal protocollo di rete usato: imparare a usarne una significa saperle usare (quasi) tutte.

### 3.1. **Le API della classe `BaseServer`**
#### 3.1.1. **Attributi principali**

`BaseServer` è la superclasse di tutti gli oggetti server del modulo: definisce l'interfaccia ma non implementa la maggior parte dei metodi, compito che spetta alle sottoclassi specifiche (`TCPServer`, `UDPServer`).

I principali **attributi** definiti per le istanze delle classi server sono:

- `BaseServer.`**`server_address`**: l'indirizzo su cui il server è in ascolto; per i protocolli Internet è una tupla `(ip, porta)`;
- `BaseServer.`**`socket`**: l'oggetto socket su cui il server ascolterà le richieste in arrivo;
- `BaseServer.`**`RequestHandlerClass`**: la classe request handler fornita al server: **un'istanza di questa classe viene creata per ogni client** che si connette al server e gestisce lo scambio dati tra il server e il client stesso.

#### 3.1.2. **Metodi principali**

I principali **metodi** definiti per le classi server sono:

- `BaseServer`.**`server_bind()`** e `BaseServer.`**`server_activate()`**: questi metodi, che **non vanno chiamati direttamente**, sono invocati dal costruttore per creare e attivare il server sull'indirizzo desiderato;

- `BaseServer.`**`handle_request()`**: metodo di alto livello usato per processare **una singola richiesta** in arrivo dal client. In realtà, `handle_request()` esegue "dietro le quinte" questa sequenza di chiamate:

    1. `BaseServer.`**`get_request()`**: **accetta** una richiesta del client e restituisce una tupla a 2 elementi contenente il nuovo oggetto socket da usare per comunicare con il client, nonchè l'indirizzo del client;

    2. `BaseServer.`**`verify_request(request, client_address)`**: restituisce semplicemente un valore booleano: se `True` la richiesta viene processata, se `False` viene rifiutata. Questo metodo, che di default restituisce sempre `True` (ovvero, accetta tutto), può essere eventualmente sovrascritto per implementare **politiche di controllo degli accessi**;

    3. `BaseServer.`**`process_request(request, client_address)`**: di norma si limita a chiamare il metodo successivo, ma può anche creare un nuovo processo o thread per gestire la richiesta (→ è esattamente ciò che fanno rispettivamente le classi `ForkingMixIn` e `ThreadingMixIn`);

    4. `BaseServer.`**`finish_request(request, client_address)`** — **crea un'istanza della classe RequestHandler** data al costruttore del server, e ne chiama il metodo `handle()` per processare effettivamente la richiesta.

Infine, per far funzionare il server in modo continuativo, si utilizzano:

- `BaseServer.`**`serve_forever(poll_interval=0.5)`**: questo metodo chiama `handle_request()` in un ciclo infinito in modo da gestire le richieste successive che arrivano finché non viene fatto un esplicito `shutdown()`. Il parametro `poll_interval` definisce ogni quanto tempo il server controlla se è stato richiesto l'arresto ed è espresso in secondi;

- `BaseServer.`**`shutdown()`**: dice al ciclo di `serve_forever()` di fermarsi, e attende che questo avvenga davvero;

- `BaseServer.`**`server_close()`** — libera le risorse del server (chiude il socket).

## 4. **La classe *Request Handler***

A differenza della classe server, che viene generalmente istanziata e utilizzata così com'è, la classe **Request Handler** deve necessariamente essere implementata per ogni applicazione che si vuole realizzare, in quanto è specifica dell'applicazione stessa.

I *request handler* svolgono la maggior parte del lavoro di elaborazione delle richieste in arrivo e sono **responsabili dell'implementazione del protocollo** al di sopra del livello dei socket, cioè del **protocollo applicativo** che definisce struttura e significato dei messaggi scambiati tra server e client.

⇒ una nuova istanza della classe Request Handler viene creata per ogni richiesta, e in generale essa:
- riceve i dati in arrivo dal canale di comunicazione;
- li interpreta e li elabora;
- predispone e invia una risposta.

### 4.1 **Metodi principali**
Nella classe Request Handler ci sono **3 metodi** che possono essere sovrascritti, e che vengono chiamati in quest'ordine:

- `RequestHandler.`**`setup()`**: può essere usato per eseguire azioni di inizializzazione: nella classe `StreamRequestHandler`, ad esempio, `setup()` crea gli oggetti file-like per leggere e scrivere sul socket (→ l'implementazione di default non fa nulla);

- `RequestHandler.`**`handle()`**: è il metodo che **deve implementare tutto il lavoro** richiesto per gestire una richiesta e pertanto **è obbligatorio sovrascriverlo** (→ l'implementazione di default non fa nulla). All'interno di `handle()` sono disponibili diversi attributi dell'istanza:
    - `self.request` è la richiesta stessa (per i servizi a stream, è un oggetto socket);
    - `self.client_address` è l'indirizzo del client;
    - `self.server` è l'istanza del server;

- `RequestHandler.`**`finish()`**: può essere usato per eseguire azioni di pulizia finale (→ l'implementazione di default non fa nulla).

## 5. **I passi per creare un server**

Creare un server richiede diversi passi:

1. derivare una classe **Request Handler**, sottoclassandola dalla classe `BaseRequestHandler`, sovrascrivendo perlomeno il suo metodo `handle()` in modo che interpreti la richiesta in arrivo, elabori i dati e invii una risposta al client;

2. istanziare una delle classi server, passandole:
   - l'indirizzo del server;
   - la classe Request Handler appena definita;

3. chiamare il metodo `handle_request()` o `serve_forever()` dell'oggetto server, per processare rispettivamente una sola o molte richieste;

4. alla fine, chiamare `server_close()` per chiudere il socket.

### Pseudocodice
```
definisci RequestHandler come sottoclasse di BaseRequestHandler
    sovrascrivi handle():
        dati = leggi da self.request
        elabora dati
        scrivi risposta su self.request

server = ServerClass(indirizzo, RequestHandler)
server.serve_forever()      # oppure server.handle_request() per una sola richiesta
# ... quando serve fermarsi:
server.shutdown()
server.server_close()
```

Confrontate questo pseudocodice con quello del server TCP "grezzo" del notebook precedente: qui non compaiono più `bind`, `listen`, `accept` — se ne occupa la classe server.

## 6. **Primo esempio: server TCP echo sincrono**

Definiamo il request handler sovrascrivendo `handle()`: `self.request` è il socket TCP connesso al client, quindi possiamo usare `recv()`/`sendall()` esattamente come nel notebook precedente.

In [ ]:
import socketserver
import threading

#----------------SERVER---------------------------
#definizione classe Request Handler
class echoRequestHandler(socketserver.BaseRequestHandler):

    def handle(self):
        data = self.request.recv(1024)
        print(f'[SERVER] ricevuto: {data.decode()}')
        self.request.sendall(data)

HOST, PORT = '127.0.0.1', 50014

# equivalente, per socketserver, di SO_REUSEADDR
socketserver.TCPServer.allow_reuse_address = True

#istanziazione classe server
server = socketserver.TCPServer((HOST, PORT), echoRequestHandler)
print(f'server avviato sulla porta {PORT}...')

#avvia server.serve_forever() in un thread daemon separato (--> necessario per l'esecuzione nel notebook)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()

In [ ]:
import socket
import time

time.sleep(0.3)

#----------------CLIENT---------------------------
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((HOST, PORT))
    s.sendall(b"ciao dal client")
    print(f"[CLIENT] risposta: {s.recv(1024).decode()}")

Fermiamo ora il server **in modo controllato**, cosa che con il socket "grezzo" del notebook precedente non era possibile senza riavviare il kernel: `serve_forever()` gira in un ciclo che controlla periodicamente se è stato chiesto uno `shutdown()`, quindi basta chiamarlo dal thread principale.

In [ ]:
server.shutdown()      # ferma il ciclo di serve_forever()
server.server_close()  # chiude il socket
print("server arrestato correttamente")

## 7. **Un modo alternativo: `StreamRequestHandler` e i file-like object**

Un modo alternativo di definire la classe Request Handler fa uso degli **stream**: oggetti "file-like" che semplificano la comunicazione fornendo la stessa interfaccia standard dei file.

Per gestire il socket come oggetto file-like, la classe Request Handler va derivata da **`socketserver.StreamRequestHandler`** invece che da `BaseRequestHandler`:

- `self.rfile` è un oggetto file-like creato dall'handler per **ricevere** dati, usando ad esempio `readline()` invece delle chiamate `recv()` "grezze";
- `self.wfile` è, allo stesso modo, un oggetto file-like usato per **scrivere** la risposta al client.

In questo esempio, il metodo `readline()` chiamerà `recv()` più volte finché non incontra un carattere di *newline*: per questo motivo il client dovrà terminare i dati inviati con `\n`.

In [ ]:
import socketserver
import threading

#----------------SERVER---------------------------
class echoRequestHandler(socketserver.StreamRequestHandler):

    def handle(self):
        data = self.rfile.readline().strip()
        print(f'[SERVER] ricevuto: {data.decode()}')
        self.wfile.write(data)

HOST, PORT = '127.0.0.1', 50015
socketserver.TCPServer.allow_reuse_address = True
server2 = socketserver.TCPServer((HOST, PORT), echoRequestHandler)

threading.Thread(target=server2.serve_forever, daemon=True).start()
print(f'server avviato sulla porta {PORT}...')

In [ ]:
import socket
import time

time.sleep(0.3)

#----------------CLIENT---------------------------
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((HOST, PORT))
    s.sendall(b"messaggio con StreamRequestHandler\n")
    print(f"[CLIENT] risposta: {s.recv(1024).decode().strip()}")

server2.shutdown()
server2.server_close()

## 8. **Server asincroni: le mix-in `ThreadingMixIn` e `ForkingMixIn`**

Le classi viste finora gestiscono le richieste **in modo sincrono**: ogni richiesta deve essere completata prima che la successiva possa iniziare (esattamente come il server "grezzo" a singolo client del notebook precedente).

Per costruire handler **asincroni**, basta includere la mix-in appropriata nella gerarchia di classi del server stesso. Le classi mix-in sovrascrivono il metodo `process_request()` per avviare un nuovo thread o processo quando una richiesta è pronta per essere gestita, in modo che l'elaborazione avvenga nel nuovo "figlio".

Per i thread si usa **`ThreadingMixIn`**:

```python
class threadedTCPServer(socketserver.ThreadingMixIn, socketserver.TCPServer):
    pass
```

**Attenzione all'ordine**: dato che sovrascrive un metodo della classe `TCPServer`, la classe `ThreadingMixIn` **deve venire prima** nella gerarchia delle classi base (Python risolve i metodi seguendo l'ordine delle classi base da sinistra a destra).

In [ ]:
import socketserver
import threading

#----------------SERVER---------------------------
class threadedEchoRequestHandler(socketserver.BaseRequestHandler):

    def handle(self):
        thread = threading.current_thread()
        while True:
            data = self.request.recv(1024)
            if not data:
                break
            print(f'[SERVER] ricevuto: {data.decode()} nel thread: {thread.name}')
            self.request.sendall(data)

class threadedTCPServer(socketserver.ThreadingMixIn, socketserver.TCPServer):
    allow_reuse_address = True

HOST, PORT = '127.0.0.1', 50016
server3 = threadedTCPServer((HOST, PORT), threadedEchoRequestHandler)

threading.Thread(target=server3.serve_forever, daemon=True).start()
print(f'server avviato sulla porta {PORT}...')


In [ ]:
import socket
import time

#----------------CLIENT---------------------------
def client(nome, messaggi):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT))
        for msg in messaggi:
            s.sendall(msg.encode())
            print(f"[{nome}] risposta: {s.recv(1024).decode()}")

time.sleep(0.3)
threading.Thread(target=client, args=("Client-1", ["primo messaggio", "ancora io"])).start()    #avvia client 1
threading.Thread(target=client, args=("Client-2", ["messaggio del secondo client"])).start()    #avvia client 2


In [ ]:
time.sleep(0.3)  # lasciamo il tempo ai client di finire, poi fermiamo il server

server3.shutdown()
server3.server_close()
print("server arrestato correttamente")


Confrontate questo esempio con il server multi-client "a mano" del notebook precedente: la logica di gestione del singolo client è praticamente identica (un ciclo che chiama `recv()` finché il client non chiude la connessione), ma qui non abbiamo dovuto scrivere né `bind`/`listen`, né il ciclo con `accept()` e la creazione esplicita del thread per ogni connessione — se ne occupa `ThreadingMixIn`. In più, a differenza di allora, **possiamo fermare il server in modo controllato** con `shutdown()`.

## 9. **E l'UDP?**

Anche per l'UDP il modulo fornisce una classe pronta, **`socketserver.UDPServer`**, con la stessa logica generale: cambia solo il modo in cui la richiesta viene esposta al request handler.

Per i server a datagramma, `self.request` non è un socket ma una **coppia** `(dati, socket)`, dove `socket` è l'oggetto socket UDP del server, da usare per rispondere con `sendto()`.

In [ ]:
import socketserver
import threading

#----------------SERVER---------------------------
class echoUDPRequestHandler(socketserver.BaseRequestHandler):

    def handle(self):
        data, sock = self.request     # per UDP: request = (dati, socket)
        print(f'[SERVER] ricevuto da {self.client_address}: {data.decode()}')
        sock.sendto(data.upper(), self.client_address)

HOST, PORT = '127.0.0.1', 50017
server4 = socketserver.UDPServer((HOST, PORT), echoUDPRequestHandler)

threading.Thread(target=server4.serve_forever, daemon=True).start()
print(f'server UDP avviato sulla porta {PORT}...')

In [ ]:
import socket
import time

time.sleep(0.3)

#----------------CLIENT---------------------------
with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
    s.sendto(b"messaggio UDP", (HOST, PORT))
    risposta, _ = s.recvfrom(1024)
    print(f"[CLIENT UDP] risposta: {risposta.decode()}")

server4.shutdown()
server4.server_close()

## 10. **Confronto `socket` vs `socketserver`: quando usare cosa**

| | `socket` (notebook precedente) | `socketserver` |
|---|---|---|
| Livello | Basso: si gestisce ogni fase a mano | Più alto: la parte di comunicazione è già pronta |
| Cosa scriviamo noi | Tutto: bind, listen, accept, cicli, thread | Solo il request handler (protocollo applicativo) |
| Concorrenza | Da implementare a mano con `threading` | Basta aggiungere `ThreadingMixIn` (o `ForkingMixIn`) |
| Arresto controllato | Difficile con `accept()` bloccante in un ciclo infinito | `shutdown()` + `server_close()` |
| Quando conviene | Quando serve il controllo più fine possibile, o per capire cosa succede "sotto" | Nella maggior parte dei casi pratici: meno codice, meno probabilità di errori |


In sintesi: aver visto prima i socket "grezzi" permette di capire *cosa fa* `socketserver` dietro le quinte; nella pratica, quando possibile, conviene usare `socketserver` perché evita di dover riscrivere sempre lo stesso codice "di infrastruttura".

## 11. **Esercizi proposti**

1. **Dal notebook precedente a `socketserver`**: riprendete l'esercizio "protocollo a comandi" (es. `ORA`, `SOMMA 3 5`, ...) e implementatelo come request handler di un `TCPServer` (senza concorrenza).
2. **UDP concorrente**: costruite una classe `threadedUDPServer(socketserver.ThreadingMixIn, socketserver.UDPServer)` e un handler che gestisca più richieste UDP "contemporaneamente" (attenzione: essendo l'UDP senza connessione, cosa significa davvero "gestire contemporaneamente" in questo caso?).
3. **Controllo di accesso**: sovrascrivete il metodo `verify_request(self, request, client_address)` in una sottoclasse di `TCPServer` in modo che rifiuti (restituendo `False`) le connessioni provenienti da un indirizzo IP a scelta, e verificate cosa succede provando a connettervi da quell'indirizzo.
4. **Broadcast tra client**: riprendete l'esercizio "avanzato" del notebook precedente (un client che invia `INVIA_A_TUTTI <messaggio>` e il server lo inoltra a tutti gli altri client connessi) e reimplementatelo con `ThreadingMixIn` + `TCPServer`. È più semplice o più complesso rispetto alla versione con i socket "grezzi"? Cosa cambia, cosa resta uguale?
5. **Discussione**: perché `ThreadingMixIn` deve essere scritto *prima* di `TCPServer` nella definizione `class threadedTCPServer(socketserver.ThreadingMixIn, socketserver.TCPServer)`? Provate a scambiare l'ordine e osservate cosa succede.